In [ ]:
%%time

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from matplotlib import colors
import os
from matplotlib import colors
from matplotlib.lines import Line2D
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

def get_dotplot_df(adata, annotation_col, genes_for_plot, expression_cutoff=0, 
                   standard_var=False, standard_method='zscore',
                   cap=None, floor=None):
    # cap / floor are quantiles (e.g. cap=0.99, floor=0.01)

    import numpy as np
    import scipy.sparse as sp
    import pandas as pd
    from scipy.stats import zscore

    # filter the anndata by genes needed for plot
    ad = adata[:, adata.var_names.isin(genes_for_plot)].copy()

    # ensure dense matrix for manipulation
    X = ad.X.toarray() if sp.issparse(ad.X) else ad.X.copy()

    # --- NEW: apply per-gene clipping ---
    if cap is not None or floor is not None:
        for i, gene in enumerate(ad.var_names):
            gene_expr = X[:, i]

            if floor is not None:
                floor_value = np.quantile(gene_expr, floor)
            else:
                floor_value = gene_expr.min()

            if cap is not None:
                cap_value = np.quantile(gene_expr, cap)
            else:
                cap_value = gene_expr.max()

            X[:, i] = np.clip(gene_expr, floor_value, cap_value)

    # convert to df
    df = pd.DataFrame(
        X,
        index=[ad.obs['cell_id'], ad.obs[annotation_col]],
        columns=ad.var_names
    )

    # get mean per group
    group_mean = df.reset_index().drop('cell_id', axis=1)\
        .groupby(annotation_col, observed=True).mean()

    # normalise/standardise the group mean
    if standard_var:
        if standard_method == 'zscore':
            group_mean = group_mean.apply(zscore, axis=0)
        elif standard_method == 'minmax':
            group_mean = (group_mean - group_mean.min(axis=0)) / (
                group_mean.max(axis=0) - group_mean.min(axis=0)
            )

    # melt df for plotting
    group_mean = group_mean.melt(ignore_index=False, var_name="feature")\
        .reset_index().set_index([annotation_col, 'feature'])\
        .rename(columns={'value': 'mean'})

    # calculate cell fraction
    cell_fraction = (df > expression_cutoff).reset_index()\
        .drop('cell_id', axis=1)\
        .groupby(annotation_col, observed=True).sum().divide(
            df.reset_index().groupby(annotation_col, observed=True)['cell_id'].count(),
            axis=0
        )\
        .melt(ignore_index=False, var_name="feature")\
        .reset_index().set_index([annotation_col, 'feature'])\
        .rename(columns={'value': 'proportion'})

    # return concatenated mean and fraction
    return pd.concat([group_mean, cell_fraction], axis=1).reset_index()

# Load AnnData object
adata=sc.read_h5ad('../../../data/GBM_LEAP_annotations/GBM_LEAP_annotations_v2.22_04_25.webatlas.h5ad')

# Set path to save figure
output_dir = '../../../data/EDFig4/'
os.makedirs(output_dir, exist_ok=True)

CPU times: user 5.58 s, sys: 13.3 s, total: 18.9 s
Wall time: 20.3 s


In [ ]:
# Normalise data
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
genes = [
    'CDH1',
    'TJP1',
    'EPCAM',
    'PARD3',
    'PARD6B',
    'COL1A1',
    'COL1A2',
    'COL3A1',
    'SNAI1',
    'SNAI2',
    'TWIST1',
    'TWIST2',
    'FOXC1',
    'FOXC2',
    'ZEB1',
    'ZEB2',
    'IFI16',
    'IL6R',
    'IL1R1',
    'HSPA1B',
    'JUN',
    'FOS',
    'VIM',
    'SOX2',
    'CTNNB1',
    'CDH2',
    'TGFB1',
    'TGFB2',
    'SMAD1',
    'SMAD2',
    'SMAD3',
    'JAK2',
    'STAT3',
    'RELA',
    'RELB',
    'NFKB1',
    'NFKB2',
    'REL'
]

In [ ]:
# Subset data
adata_subset = adata[adata.obs['cell_status']=='Malignant',:].copy()

In [ ]:
dp_df = get_dotplot_df(
    adata_subset,
    'annotation_coarse',
    genes,
    standard_var=True,
    standard_method='minmax',
    cap=1,
    floor=0
)

In [ ]:
cell_states = [
 'Hypoxic',
 'Gliosis-like',
 'AC-gliosis-like',
 'AC-progenitor-like',
 'Proliferative',

 'NPC-neuronal-like',
 'OPC-neuronal-like',
 'OPC-like',
 'OPC-NPC-like',
]

In [ ]:
dp_df_cell_states = dp_df[dp_df['annotation_coarse'].isin(cell_states)]
dp_df_cell_states['annotation_coarse'] = dp_df_cell_states['annotation_coarse'].cat.remove_unused_categories()
dp_df_cell_states['annotation_coarse'] = dp_df_cell_states['annotation_coarse'].cat.set_categories(cell_states)

dp_df_cell_states['feature'] = dp_df_cell_states['feature'].astype('category')
dp_df_cell_states['feature'] = dp_df_cell_states['feature'].cat.set_categories(genes)

In [ ]:
# --- FILTER: drop low-proportion dots ---
dp_df_plot = dp_df_cell_states.copy()

# Color scaling
vmax = np.quantile(dp_df_plot['mean'], 0.99)
#norm = colors.TwoSlopeNorm(vmin=0, vcenter=1.6, vmax=2)
cmap = plt.get_cmap('Reds')

# --- Dot-size scaling ---
size_min, size_max = 0, 240
pmin, pmax = dp_df_plot['proportion'].min(), dp_df_plot['proportion'].max()

prop_scaled = (dp_df_plot['proportion'] - pmin) / (pmax - pmin)
prop_scaled = np.clip(prop_scaled, 0, 1)
size_exp = 1.8

dp_df_plot['dot_area'] = size_min + (size_max - size_min) * (prop_scaled ** size_exp)

plt.figure(figsize=(10, 6))

ax = sns.scatterplot(
    data=dp_df_plot,
    x='feature',
    y='annotation_coarse',
    hue='mean',
    size='dot_area',
    sizes=(size_min, size_max),
    palette='Reds',
    #hue_norm=norm,
    edgecolor='black',
    linewidth=0.4,
    legend=False
)

# Grid lines
ax.set_axisbelow(True)
ax.grid(True, axis='x', color='lightgrey', linewidth=0.4)
ax.grid(True, axis='y', color='lightgrey', linewidth=0.4)

plt.xticks(rotation=90)
plt.title('EMT marker genes (snRNA-seq)')

# --- COLOR LEGEND ---
color_values = [-0.8, 0.0, 0.8, 1.6]
color_handles = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='',
        markersize=8,
        markerfacecolor=cmap(norm(v)),
        markeredgecolor='black',
        markeredgewidth=0.6,
        label=f'{v:g}'
    )
    for v in color_values
]

# --- SIZE LEGEND ---
size_values = [0.4, 0.6, 0.8]
size_handles = []
for v in size_values:
    v_scaled = (v - pmin) / (pmax - pmin)
    v_scaled = np.clip(v_scaled, 0, 1)
    area = size_min + (size_max - size_min) * (v_scaled ** size_exp)
    size_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            linestyle='',
            markersize=np.sqrt(area),
            markerfacecolor='lightgrey',
            markeredgecolor='black',
            label=f'{v:.2f}'
        )
    )

# Combine legends
handles = (
    [Line2D([], [], linestyle='', label='Mean expression')] +
    color_handles +
    [Line2D([], [], linestyle='', label='Proportion')] +
    size_handles
)

labels = [h.get_label() for h in handles]

ax.legend(
    handles,
    labels,
    bbox_to_anchor=(0.5, -0.45),
    loc='upper center',
    ncol=4,
    frameon=False
)

ax.set_xlabel('')
ax.set_ylabel('Malignant cell states')

plt.show()
dp_df.to_csv(f'{output_dir}/EDFig4a.csv')